In [0]:

# ==============================================================================
# PWG PIPELINE 3.0 - FULL PIPELINE VALIDATION SCRIPT (After Batch 3)
# ==============================================================================

catalog = "charles_schwab_retailbrokerage_dev_team_lemma"


# Dictionary of all tables across all layers with their EXACT expected counts
validation_targets = {
    # ---------------------------------------------------------
    # BRONZE LAYER (Append-Only Raw Archive)
    # ---------------------------------------------------------
    "bronze.batchdate": 3,
    "bronze.date": 26028,
    "bronze.time": 86465,
    "bronze.statustype": 6,
    "bronze.taxrate": 320,
    "bronze.industry": 102,
    "bronze.tradetype": 5,
    "bronze.finwire": 471446,          # Includes header lines
    "bronze.dailymarket": 5285024,
    "bronze.hr": 50000,
    "bronze.customermgmt": 50000,
    "bronze.customer": 100,            # B2 + B3 CDC
    "bronze.prospect": 149820,         # Cumulative 3 batches
    "bronze.watchhistory": 3013989,
    "bronze.account": 200,             # B2 + B3 CDC
    "bronze.cashtransaction": 1204943,
    "bronze.trade": 1304387,           # Cumulative raw CDC rows
    "bronze.trade_history": 3267433,
    "bronze.holdinghistory": 1206578,

    # ---------------------------------------------------------
    # SILVER LAYER (Cleansed / Deduplicated)
    # ---------------------------------------------------------
    "silver.batchdate": 3,
    "silver.date": 25933,
    "silver.time": 86400,
    "silver.statustype": 6,
    "silver.taxrate": 320,
    "silver.industry": 102,
    "silver.tradetype": 5,
    "silver.broker": 50000,
    "silver.company": 4596,
    "silver.security": 7598,
    "silver.financial": 457025,
    "silver.markethistory": 5285024,
    "silver.customer": 15380,
    "silver.watches": 3013989,
    "silver.prospect": 50059,
    "silver.account": 30610,
    "silver.cash_transactions": 1204943,
    "silver.trade": 1302248,
    "silver.trade_history": 3267433,
    "silver.holdings": 1206578,

    # ---------------------------------------------------------
    # GOLD LAYER (Analytics Star Schema)
    # ---------------------------------------------------------
    "gold.dim_date": 25933,
    "gold.dim_time": 86400,
    "gold.dim_broker": 14239,          # Filtered to job code 314
    "gold.dim_company": 5000,          # SCD-2 Expanded
    "gold.dim_security": 8658,         # SCD-2 Expanded
    "gold.dim_customer": 21890,        # SCD-2 Expanded
    "gold.dim_account": 56392,         # SCD-2 Expanded
    "gold.dim_prospect": 49940,        # Current state only
    "gold.dim_trade": 1302248,
    "gold.financial": 457025,
    "gold.fact_markethistory": 5285024,
    "gold.fact_watches": 2412745,      # Active watches only
    "gold.fact_cash_transactions": 1204943,
    "gold.fact_cash_balances": 1088273,# Aggregated derived table
    "gold.fact_trade_history": 3267433,
    "gold.fact_holdings": 1206578
}

print(f"{'SCHEMA.TABLE':<32} | {'EXPECTED':<10} | {'ACTUAL':<10} | {'STATUS'}")
print("=" * 80)

passed_count = 0
built_tables = 0
missing_tables = 0

for schema_table, expected in validation_targets.items():
    full_table_path = f"{catalog}.{schema_table}"
    
    # Add a visual separator between layers for readability
    if schema_table == "bronze.batchdate":
        print(f"\n--- BRONZE LAYER ---")
    elif schema_table == "silver.batchdate":
        print(f"\n--- SILVER LAYER ---")
    elif schema_table == "gold.dim_date":
        print(f"\n--- GOLD LAYER ---")
    
    try:
        # Check if table exists in the Unity Catalog
        if spark.catalog.tableExists(full_table_path):
            built_tables += 1
            actual = spark.read.table(full_table_path).count()
            
            # Check if actual matches expected
            if actual == expected:
                status = "✅ PASS"
                passed_count += 1
            else:
                status = "⚠️ FAIL (Mismatch)"
        else:
            actual = "---"
            status = "⏳ Missing"
            missing_tables += 1
            
        print(f"{schema_table:<32} | {expected:<10} | {actual:<10} | {status}")
        
    except Exception as e:
        print(f"{schema_table:<32} | {expected:<10} | {'ERROR':<10} | ❌ Read Error")

print("=" * 80)
print(f"VALIDATION SUMMARY:")
print(f"Total Tables Checked : {len(validation_targets)}")
print(f"Tables Present       : {built_tables}")
print(f"Tables Missing       : {missing_tables}")
print(f"Tables Passed Exact  : {passed_count} / {built_tables}")
print("=" * 80)